# Capítulo 5: Programando sin Programar (AutoML y Herramientas No-Code)

> *"AutoML es como un GPS: te lleva al destino, pero no sabes por dónde pasaste."*

Este notebook acompaña el Capítulo 5 del libro *Ciencia de Datos sin Filtros*. Aquí exploraremos:

1. Carga y análisis del dataset de SaaS Churn
2. Análisis con prompts de ChatGPT/Claude
3. Construcción de modelo con H2O AutoML
4. Interpretación de resultados
5. Comparación manual vs AutoML
6. Ética: ¿Entiendes tu modelo?

## Celda 1: Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print('Librerías cargadas correctamente.')

## Celda 2: Carga de datos de SaaS Churn

El dataset contiene 2000 clientes SaaS con información sobre uso, soporte, facturación y si abandonaron la plataforma.

In [ ]:
df = pd.read_csv('../datos/datos_saas_churn.csv')

print(f'Dimensiones: {df.shape[0]} filas, {df.shape[1]} columnas')
print(f'\nPrimeras filas:')
df.head(10)

In [ ]:
print('=== INFORMACIÓN GENERAL ===')
print(f'\nTipos de datos:')
print(df.dtypes)
print(f'\nValores nulos:')
print(df.isnull().sum())
print(f'\nEstadísticas descriptivas:')
df.describe()

In [ ]:
print('=== DISTRIBUCIÓN DE CHURN ===')
churn_counts = df['churned'].value_counts()
churn_pct = df['churned'].value_counts(normalize=True) * 100

print(f'No churn (0): {churn_counts[0]} clientes ({churn_pct[0]:.1f}%)')
print(f'Churn (1):    {churn_counts[1]} clientes ({churn_pct[1]:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['churned'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribución de Churn')
axes[0].set_ylabel('Cantidad de clientes')
axes[0].set_xticklabels(['Permanece (0)', 'Churn (1)'], rotation=0)

df.groupby('plan')['churned'].mean().sort_values(ascending=False).plot(kind='bar', ax=axes[1], color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[1].set_title('Tasa de Churn por Plan')
axes[1].set_ylabel('Proporción de churn')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

df.groupby('industry')['churned'].mean().sort_values(ascending=False).plot(kind='bar', ax=axes[2], color=['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db'])
axes[2].set_title('Tasa de Churn por Industria')
axes[2].set_ylabel('Proporción de churn')
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Celda 3: Análisis con prompts de ChatGPT

En esta sección simulamos el flujo de trabajo con un LLM. En la práctica, pegarías los siguientes prompts en ChatGPT o Claude:

### Prompt 1: Análisis exploratorio
```
Tengo un dataset de rotación de clientes SaaS con 2000 filas.
Las columnas son: customer_id, company_size, plan, monthly_usage,
support_tickets, contract_length, monthly_revenue, churned,
days_since_last_login, num_users, industry.

Haz un análisis exploratorio completo. Identifica:
1. Distribución de la variable objetivo (churned)
2. Variables numéricas con mayor correlación con churn
3. Patrones por plan y tamaño de empresa
4. Valores atípicos o anomalías
5. Recomendaciones de features para un modelo predictivo
```

### Prompt 2: Código para modelo
```
Genera código en Python para un Gradient Boosting que prediga churn.
Incluye preprocesamiento, validación cruzada de 5 folds,
y métricas AUC-ROC, precision, recall, F1.
```

### Lo que el LLM hace bien y mal

In [ ]:
llm_assessment = pd.DataFrame({
    'Capacidad': [
        'Generar código funcional',
        'Explicar conceptos',
        'Identificar patrones simples',
        'Crear visualizaciones',
        'Entender contexto de negocio',
        'Garantizar corrección estadística',
        'Detectar sesgos en datos',
        'Reemplazar validación humana'
    ],
    'Calificación': [4, 5, 3, 4, 1, 2, 2, 1],
    'Nivel': ['Bueno', 'Excelente', 'Regular', 'Bueno', 'Malo', 'Malo', 'Malo', 'Malo']
})

print('=== EVALUACIÓN DE LLMs COMO HERRAMIENTA DE ANÁLISIS ===')
print('(Escala: 1=Malo, 5=Excelente)\n')
for _, row in llm_assessment.iterrows():
    bar = '█' * row['Calificación'] + '░' * (5 - row['Calificación'])
    print(f"{row['Capacidad']:<40} [{bar}] {row['Calificación']}/5 ({row['Nivel']})")

print('\n' + '='*60)
print('REGLA: Siempre verifica el código generado antes de usarlo.')
print('Los LLMs son asistentes, no expertos.')
print('='*60)

## Celda 4: Construcción de modelo con H2O AutoML

### Instalación y configuración

```bash
pip install h2o
```

### Código H2O AutoML

El siguiente código demuestra el flujo completo de H2O AutoML. **Nota**: H2O requiere Java instalado en el sistema.

In [ ]:
# ============================================================
# H2O AutoML - Ejemplo completo
# ============================================================
# Descomentar las líneas de abajo si H2O está instalado

h2o_code = '''
import h2o
from h2o.automl import H2OAutoML

# Inicializar H2O
h2o.init(nthreads=-1, max_mem_size='4G')

# Cargar datos
df_h2o = h2o.import_file('../datos/datos_saas_churn.csv')

# Definir variables
target = 'churned'
features = ['company_size', 'plan', 'monthly_usage', 'support_tickets',
            'contract_length', 'monthly_revenue', 'days_since_last_login',
            'num_users', 'industry']

# Convertir target a factor (clasificación)
df_h2o[target] = df_h2o[target].asfactor()

# Split train/test
train, test = df_h2o.split_frame(ratios=[0.8], seed=42)

print(f'Train: {train.nrows} filas')
print(f'Test:  {test.nrows} filas')

# Configurar AutoML
aml = H2OAutoML(
    max_models=10,
    seed=42,
    max_runtime_secs=300,
    balance_classes=True,
    sort_metric='AUC'
)

# Entrenar
aml.train(x=features, y=target, training_frame=train)

# Leaderboard
lb = aml.leaderboard
print('\n=== LEADERBOARD ===')
print(lb.head(10))

# Mejor modelo
best_model = aml.leader
print(f'\nMejor modelo: {best_model.model_id}')
print(f'AUC en train: {best_model.auc()}')

# Evaluación en test
perf = best_model.model_performance(test)
print(f'AUC en test:  {perf.auc()}')
print(f'LogLoss:     {perf.logloss()}')

# Importancia de variables
print('\n=== IMPORTANCIA DE VARIABLES ===')
print(best_model.varimp())

# SHAP values (si el modelo lo soporta)
# shap_values = best_model.shap_summary_plot(test)

# Guardar modelo
model_path = h2o.save_model(best_model, path='./modelo_h2o_churn')
print(f'\nModelo guardado en: {model_path}')

# Cerrar H2O
h2o.cluster().shutdown()
'''

print('=== CÓDIGO H2O AutoML ===')
print('Copia y pega este código en un notebook con H2O instalado.\n')
print(h2o_code)

## Celda 5: Interpretación de resultados

### Qué significa el leaderboard

Cuando H2O te devuelve un leaderboard como este:

| Model | AUC | LogLoss |
|-------|-----|----------|
| GBM_grid_1 | 0.892 | 0.342 |
| XGBoost_grid | 0.887 | 0.351 |
| DRF | 0.865 | 0.389 |

Necesitas responder:
1. ¿Por qué GBM ganó? ¿Es consistente o es ruido?
2. ¿La diferencia entre 0.892 y 0.887 es significativa?
3. ¿El modelo es interpretable para el negocio?

In [ ]:
print('=== GUÍA DE INTERPRETACIÓN ===')
print()
print('1. AUC > 0.85: Bueno para un primer modelo')
print('   - Pero NO significa que el modelo esté listo para producción')
print()
print('2. Si AUC_train >> AUC_test: Sobreajuste')
print('   - Train=0.95, Test=0.78 → problema serio')
print('   - Train=0.89, Test=0.87 → aceptable')
print()
print('3. Importancia de variables:')
print('   - Si "days_since_last_login" es #1: Tiene sentido (clientes inactivos churnan)')
print('   - Si "customer_id" es #1: ¡ALERTA! El modelo está memorizando IDs')
print('   - Si "industry" es bajo: ¿Los datos industriales son relevantes?')
print()
print('4. Preguntas que AutoML NO responde:')
   questions = [
    '¿Por qué este algoritmo y no otro?',
    '¿Qué features creó automáticamente?',
    '¿El modelo es justo para todas las industrias?',
    '¿Qué pasa si los patrones de uso cambian?',
    '¿Puedo explicar una predicción individual?'
]
for q in questions:
    print(f'   - {q}')

print()
print('5. La trampa del "no entender":')
print('   Si no puedes responder estas preguntas, no deberías')
print('   desplegar el modelo en producción.')

## Celda 6: Comparación manual vs AutoML

Veamos la diferencia entre construir un modelo manualmente y usar AutoML.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Preparar datos manualmente
df_model = df.copy()

# Encoding de variables categóricas
le_dict = {}
categorical_cols = ['company_size', 'plan', 'industry']
for col in categorical_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    le_dict[col] = le

# Features y target
X = df_model.drop(['customer_id', 'churned'], axis=1)
y = df_model['churned']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Churn rate train: {y_train.mean():.2%} | Test: {y_test.mean():.2%}')

In [ ]:
# ============================================================
# MODELO 1: Regresión Logística (explicable)
# ============================================================
print('=== REGRESIÓN LOGÍSTICA ===')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.3f}')
print(f'\nCoeficientes (interpretación directa):')
for name, coef in zip(X.columns, lr.coef_[0]):
    direction = '+' if coef > 0 else '-'
    print(f'  {name:<25} {direction}{abs(coef):.4f}')
print(f'  (Valores positivos → mayor probabilidad de churn)')

print(f'\nMatriz de confusión:')
cm = confusion_matrix(y_test, y_pred_lr)
print(f'  TN={cm[0,0]:4d}  FP={cm[0,1]:4d}')
print(f'  FN={cm[1,0]:4d}  TP={cm[1,1]:4d}')

In [ ]:
# ============================================================
# MODELO 2: Random Forest
# ============================================================
print('=== RANDOM FOREST ===')
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(X_train, y_train)

y_prob_rf = rf.predict_proba(X_test)[:, 1]
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.3f}')

print(f'\nImportancia de variables:')
importance_df = pd.DataFrame({
    'Variable': X.columns,
    'Importancia': rf.feature_importances_
}).sort_values('Importancia', ascending=False)
for _, row in importance_df.iterrows():
    bar = '█' * int(row['Importancia'] * 50)
    print(f"  {row['Variable']:<25} {bar} {row['Importancia']:.3f}")

# ============================================================
# MODELO 3: Gradient Boosting
# ============================================================
print('\n=== GRADIENT BOOSTING ===')
gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

y_prob_gb = gb.predict_proba(X_test)[:, 1]
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_gb):.3f}')

In [ ]:
# ============================================================
# COMPARACIÓN FINAL
# ============================================================
print('=' * 65)
print('COMPARACIÓN: MODELO MANUAL vs AutoML')
print('=' * 65)

results = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Random Forest', 'Gradient Boosting', 'H2O AutoML (ejemplo)'],
    'AUC-ROC': [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_gb),
        0.892  # valor típico de H2O
    ],
    'Explicable': ['Alta', 'Media', 'Media', 'Baja'],
    'Tiempo de desarrollo': ['30 min', '1 hora', '1 hora', '5 min'],
    'Control total': ['Sí', 'Sí', 'Sí', 'No']
})

print(results.to_string(index=False))

print('\n' + '=' * 65)
print('LECCIÓN:')
print('  AutoML es más rápido, pero no necesariamente mejor.')
print('  Un Gradient Boosting bien tuneado puede igualar a AutoML.')
print('  Y lo más importante: con código, SABES qué está pasando.')
print('=' * 65)

## Celda 7: Ética - ¿Entiendes tu modelo?

### La prueba final

Si no puedes responder estas preguntas sobre tu modelo, **no deberías desplegarlo en producción**.

In [ ]:
print('╔' + '═' * 63 + '╗')
print('║' + '  CHECKLIST ÉTICO ANTES DE DESPLEGAR UN MODELO'.center(63) + '║')
print('╠' + '═' * 63 + '╣')

checklist = [
    ('DATOS', [
        '¿El dataset es representativo de la población afectada?',
        '¿Hay variables protegidas (género, raza, edad)?',
        '¿Los datos de entrenamiento son recientes?',
        '¿Se manejaron valores faltantes correctamente?',
    ]),
    ('MODELO', [
        '¿Puedo explicar por qué el modelo toma una decisión?',
        '¿El modelo es justo para todos los subgrupos?',
        '¿Validé con datos que el modelo nunca vio?',
        '¿El rendimiento es estable en diferentes períodos?',
    ]),
    ('PRODUCCIÓN', [
        '¿Hay un humano en el loop para decisiones de alto impacto?',
        '¿Tengo un plan de monitoreo de drift?',
        '¿Documenté las limitaciones del modelo?',
        '¿Tengo un plan de rollback si el modelo falla?',
    ]),
]

for category, items in checklist:
    print(f'║  [{category}]' + ' ' * (60 - len(category)) + '║')
    for item in items:
        truncated = item[:58] if len(item) > 58 else item
        print(f'║    [ ] {truncated}' + ' ' * (54 - len(truncated) - 4) + '║')
    print('║' + ' ' * 63 + '║')

print('╚' + '═' * 63 + '╝')

print('\nSi algún checkbox queda sin marcar, NO despleiegues el modelo.')
print('La precisión sin explicabilidad es irresponsabilidad.')

In [ ]:
# ============================================================
# DEMOSTRACIÓN: La trampa de no entender el modelo
# ============================================================

print('=== LA TRAMPA: MODELO PRECISO PERO ILEGAL ===')
print()
print('Caso real: Una empresa de telecom creó un modelo de churn con AutoML.')
print('El modelo tenía 92% de accuracy. Lo pusieron en producción.')
print()
print('Seis meses después, descubrieron que el modelo había aprendido')
print('a predecir churn basándose en el CÓDIGO POSTAL del cliente.')
print()
print('¿Por qué? Porque los clientes de ciertas zonas tenían peor')
print('servicio al cliente, y eso correlacionaba con churn.')
print()
print('El modelo NO estaba prediciendo churn.')
print('Estaba prediciendo INFRAESTRUCTURA DEFICIENTE.')
print()
print('Consequences:')
print('  1. Discriminación geográfica (ilegal en muchos países)')
print('  2. Ofertas de retención dirigidas al grupo equivocado')
print('  3. Pérdida de clientes buenos en zonas "malas"')
print('  4. Daño reputacional cuando se filtró la información')
print()
print('LECCIÓN: Un modelo que no entiendes es un modelo peligroso.')
print('=' * 60)

In [ ]:
# ============================================================
# RESUMEN DEL CAPÍTULO
# ============================================================

print('╔' + '═' * 63 + '╗')
print('║' + '  RESUMEN DEL CAPÍTULO 5'.center(63) + '║')
print('╠' + '═' * 63 + '╣')
print('║' + ' ' * 63 + '║')
print('║  AutoML es un GPS: te lleva al destino, pero no' + ' ' * 12 + '║')
print('║  sabes por dónde pasaste.' + ' ' * 36 + '║')
print('║' + ' ' * 63 + '║')
print('║  Herramientas no-code (KNIME, Power Query):' + ' ' * 16 + '║')
print('║    ✓ Excelentes para exploración y prototipos' + ' ' * 14 + '║')
print('║    ✗ Peligrosas para producción sin supervisión' + ' ' * 12 + '║')
print('║' + ' ' * 63 + '║')
print('║  La regla de oro:' + ' ' * 44 + '║')
print('║    SI EL MODELO AFECTA PERSONAS → USA CÓDIGO' + ' ' * 15 + '║')
print('║    Y ENTIENDE CADA LÍNEA.' + ' ' * 35 + '║')
print('║' + ' ' * 63 + '║')
print('║  La trampa: conducir automático sin saber frenar.' + ' ' * 11 + '║')
print('║  Funciona el 99% del tiempo.' + ' ' * 32 + '║')
print('║  El 1% restante, no tienes habilidades para reaccionar.' + ' ' * 5 + '║')
print('║' + ' ' * 63 + '║')
print('╚' + '═' * 63 + '╝')